# Fitting the CMA Model to Kaggle Brist1D CGM Data

Reference: https://www.kaggle.com/competitions/brist1d/data

## Dataset description (from Kaggle)

The dataset is from a study that collected data from young adults in the UK with type 1 diabetes, who used a continuous glucose monitor (CGM), an insulin pump and a smartwatch. These devices collected blood glucose readings, insulin dosage, carbohydrate intake, and activity data. The data collected was aggregated to five-minute intervals and formatted into samples. Each sample represents a point in time and includes the aggregated five-minute intervals from the previous six hours. The aim is to predict the blood glucose reading an hour into the future, for each of these samples.

The training set takes samples from the first three months of study data from nine of the participants and includes the future blood glucose value. These training samples appear in chronological order and overlap. The testing set takes samples from the remainder of the study period from fifteen of the participants (so unseen participants appear in the testing set). These testing samples do not overlap and are in a random order to avoid data leakage.

__Complexities to be aware of:__

+ this is medical data so there are missing values and noise in the data
+ the participants did not all use the same device models (CGM, insulin pump and smartwatch) so there may be differences in the collection method of the data
+ some participants in the test set do not appear in the training set


In [ ]:
#%%sh

### Install dependencies for conda:

# nix-shell -p micromamba

# micromamba install -c conda-forge ipykernel pandas --force-reinstall
#"$HOME/.nix-profile/bin/micromamba" run pip install -e '../../' --upgrade --force-reinstall


## Load the Kaggle Brist1D Dataset

In [ ]:
import pandas as pd
import numpy as np
from pfun_cma_model.misc.pathdefs import PFunDataPaths
import duckdb
from datetime import datetime
import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Ensure your kaggle.json is correctly configured before running this command
# We use the built-in paths helper to download the Brist1D dataset via the Kaggle API and save it as a fast Parquet file
paths = PFunDataPaths()
paths.download_kaggle_brist1d(overwrite=True)

# Load CGM data from the downloaded Brist1D dataset
df = pd.read_parquet(paths.brist1d_data_fpath)
df

In [ ]:
display(df.dtypes.to_frame().T)

## Preprocess the data

The model requires 'time' in UTC format, and typically 'G' (glucose) and 'sg' (sensor glucose) columns

In the Kaggle dataset, 'time' is just HH:MM:SS, and 'bg-0:00' is the current blood glucose reading.
We synthesize full dates starting from today for demonstration purposes.

In [ ]:
from typing import Sequence


# For this example, we will focus on a single participant's continuous glucose trace
p_num = 'p02'
participant_df = df[df['p_num'] == p_num].copy().reset_index(drop=True)

def create_synthetic_datetime(time: pd.Series) -> Sequence['datetime64[us, UTC]']:
    today_string = pd.Timestamp.today().strftime('%Y-%m-%d') 
    datetime_string = today_string + ' ' + time.astype(str)
    datetime_vector = pd.to_datetime(datetime_string).dt.tz_localize('UTC')
    return datetime_vector

participant_df['time'] = create_synthetic_datetime(participant_df["time"])
participant_df.sort_values(by="time", inplace=True)
participant_df.set_index("time", drop=False, inplace=True)
participant_df['displayTime'] = participant_df['time']
participant_df['systemTime'] = participant_df['time']

# Combine sources of glucose data (shifting time index to match actual measurement time)
participant_df['G'] = participant_df['bg-0:00']  # Actual glucose value at the time index
bg_columns = [c for c in participant_df.columns if 'bg' in c]
time_vec = pd.to_datetime(participant_df["time"]).drop_duplicates(keep='first')
for col in bg_columns:
    print(f"Incorporating {col}...")
    delta = col.replace("bg", "") + ":00"
    shifted_time = time_vec + pd.Timedelta(delta)
    shifted_time.name = "shifted_time"
    g_vector = participant_df[col].drop_duplicates().resample('5min').mean()
    bg_new = pd.DataFrame({"G": g_vector}, index=shifted_time).dropna()
    participant_df = participant_df.combine_first(bg_new)
    print(f"N_missing=", participant_df["G"].isna().sum())


In [ ]:
# complete cgm dataset
participant_df['sg'] = participant_df['G']
cgm_data = participant_df[['time', 'G', 'sg']]
# handle duplicate time indices
cgm_data.index.name = "original_index"
cgm_data = cgm_data.resample('5min').mean()

# Checking the structure of the loaded data
print(cgm_data.head(), end="\n")
print("Shape:\n", cgm_data.shape, end="\n")
print("Data types:\n", cgm_data.dtypes, end="\n")
print(cgm_data.describe(), end="\n")


## Plot the selected CGM data

In [ ]:
from pfun_common.plot import lineplot as pfun_lineplot
original_sg = cgm_data['sg'].copy()
cgm_data['sg'] = original_sg * 18
cgm_data['G'] = original_sg * 18
ax = pfun_lineplot(cgm_data, tcol="time", ycol="sg")
ax.set_title(f"BG Data ({p_num})")
ax.figure.set_size_inches(10, 4)

## Perform the model fit

+ Our fitting procedure is sensitive to periodic noise.
+ Some hyperparameter optimization will help find an ROI in the parameter space.

In [ ]:
#from scipy.optimize import minimize
from pfun_cma_model.engine.fit import fit_model

# Fit the model to the CGM data

# Bounds-constrained minimization:
#   Nelder-Mead, L-BFGS-B, TNC, SLSQP, Powell

# Define the hyperparameters to vary
algorithms = (
    "Nelder-Mead",
    "L-BFGS-B",
    # "TNC",
    # "SLSQP",
    # "Powell"
)
nr_timepoints = (24, 48, 96, 192, 512, 720, 1024, 4096, )
meal_freq_estimates = ('2h', '4h', '6h', '8h', )

residuals = {}

def get_residual(result) -> float:
    residual = result.infodict["result"].fun
    return residual

for N in nr_timepoints:  # number of time points (resampling)
    for tm_freq in meal_freq_estimates:  # meal frequency estimate
        for algo in algorithms:  # bounds-constrained minimization algorithm
            try:
                result = fit_model(
                    cgm_data,
                    tcol='time',
                    ycol='G',
                    tm_freq=tm_freq,
                    N=N,
                    curve_fit_kwargs={
                        "method": f"{algo}",
                    }
                )
            except RuntimeError as exception:
                logging.warning(
                    "\n({%s %s, %s) Failed to converge.",
                    N, algo, tm_freq,
                    exc_info=False
                )
            else:
                result_key = (N, algo, tm_freq)
                residual = get_residual(result)
                residuals[result_key] = result
                print(f"{str(result_key)} residual = {residual:.4f}")


### Analyze hyperparameters

In [ ]:
resid_by_N = map(lambda pair: (pair[0][0], get_residual(pair[1])), residuals.items())
df_resid_by_N = pd.DataFrame(resid_by_N, columns=["N", "residual"])
ax = df_resid_by_N.plot(x="N", y="residual", linestyle='', marker='o')
ax.set_ylabel("Residual")

In [ ]:
resid_by_Algo = map(lambda pair: (pair[0][1], get_residual(pair[1])), residuals.items())
df_resid_by_Algo = pd.DataFrame(resid_by_Algo, columns=["Algo", "residual"])
ax = df_resid_by_Algo.boxplot(by="Algo")

In [ ]:
resid_by_tMf = map(lambda pair: (pair[0][2], get_residual(pair[1])), residuals.items())
df_resid_by_tMf = pd.DataFrame(resid_by_tMf, columns=["Meal Freq", "residual"])
ax = df_resid_by_tMf.boxplot(by="Meal Freq")

In [ ]:
def weighted_objective_function(result):
    residual = get_residual(result)
    N = result.soln.shape[0]
    weighted_objective = residual / ((N + 1)**(0.5))
    return weighted_objective


best_algorithm, model_results = min(
    residuals.items(),
    key=lambda pair: weighted_objective_function(pair[1])
)
best_residual = get_residual(model_results)
print(f"Algorithm: '{best_algorithm}' ({best_residual:.3f})")

model_results.soln.head()
model_results.soln.plot(x="t", y="G")

print("Model Parameters:")
print(model_results.popt_named)
print("\nModel Fit Results Info:")
print(model_results.infodict)
print(model_results.infodict['message'])


In [ ]:
# Displaying results
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
# Use formatted_data for normalized raw glucose values (scaled to [0, 2])
plt.plot(model_results.formatted_data['t'], model_results.formatted_data['G'], label='CGM Data (normalized)', marker='o', linestyle='none', alpha=0.5)

# The model_results.soln dataframe contains the fitted curve (indexed by timedelta in hours)
plt.plot(model_results.soln["t"], model_results.soln['G'], label='Fitted CMA Model', color='red', linewidth=2)

plt.title(f'Participant {p_num}, CMA Model Fit\nResidual = {best_residual:.3f}\nNormalized BG over 24-hour time')
plt.xlabel('Time (hours)')
plt.ylabel('Glucose (normalized)')
plt.legend()
plt.show()


## Posthoc Analysis

The PFun CMA model effectively characterizes the dynamic continuous glucose monitor (CGM) traces by adjusting specific physiological parameters (`taup`, `taug`, `Cm`).

By applying the model directly to the Kaggle Brist1D dataset, we observe that the CMA model is capable of fitting its compartmental equations to the raw glucose (`G`) values, providing the solid 'red' regression line above. The `infodict` generated through the fitting process yields vital metrics, like termination messages, to confirm the algorithm successfully converged on optimized coefficients.

**Key Highlights:**
- The fitting process expects timestamp columns (`time` by default) in **UTC** format, which we handled by appending arbitrary UTC dates to the `HH:MM:SS` time values provided in the competition dataset.
- Additional preprocessing steps are often handled directly through `fit_model` calling `format_data`.
- By observing the parameter bounds and bounds conditions, one can tune the CMA model further to adjust to real physiological profiles seen across different participants.

### Save rendered html

In [ ]:
%%sh

uv run jupyter nbconvert --to html kaggle-brist1d-fit-model-example.ipynb --output kaggle-brist1d-fit-model-example.html
cp kaggle-brist1d-fit-model-example.html ../../pfun_cma_model/static/notebooks/kaggle-brist1d-fit-model-example.html